# Phase 4 — Evaluation Datasets from Production Traces

Databricks AI Evals Tutorial | Phase 4 of 10

Every dataset written by hand shares one flaw: **it can only contain questions its author
thought of.** `eval_dataset.py` was written by the same person who built the agent, which
means it tests the agent against that person's model of what users want — not against what
users actually do.

The OpenAI evaluation guide lists "biased datasets misrepresenting production traffic" as a
core anti-pattern. This phase is the fix: mine real traffic, find what the curated set
never covered, and turn it into a managed evaluation dataset.

> **Order note.** This phase was built after Phase 5. Nothing here is a prerequisite for
> regression detection — in fact Phase 5 deliberately uses the *fixed* curated set, because
> comparing two prompt versions against a dataset that changed between runs measures
> nothing. Mined data **augments** the curated set; it doesn't replace it.

## The trap to name before starting: circular ground truth

The appealing shortcut, once you have production traces, is to use the agent's own recorded
responses as the expected answers. Every trace already has an input *and* an output, so it
looks like a free labelled dataset.

**It isn't.** A trace records what the agent *did*, not what it *should have done*. Build
expectations from the agent's own outputs and you get a dataset the agent passes by
construction — it can only ever agree with itself. Worse, you've frozen today's behaviour,
bugs included, as the definition of correct, which makes regression detection structurally
impossible: any change from current behaviour now reads as a *failure*, and any current
failure reads as *correct*.

So the honest summary of this phase:

> **Mining traces gives you realistic inputs for free. Expectations remain manual work.**

What you get is the *distribution* — which is the expensive half to guess and the part
hand-writing gets wrong.

## Step 0 — Setup

Evaluation datasets, like the Prompt Registry in Phase 5, require a **SQL-backed** tracking
store (SQLite, PostgreSQL, MySQL, MSSQL). They are not available on a `file://` store —
which is the second reason this track uses SQLite locally.

In [ ]:
# ============ SETUP ============
import os
from collections import Counter

import mlflow

TRACKING_MODE = os.environ.get("MLFLOW_TRACKING_MODE", "local")
os.environ.setdefault("TELCOASSIST_PROVIDER", "databricks")

if TRACKING_MODE == "databricks":
    mlflow.set_tracking_uri("databricks")
    EXPERIMENT = "/Shared/telcoassist-evals"
else:
    # Evaluation Datasets require a SQL backend -- not available on a file:// store.
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
    EXPERIMENT = "telcoassist-evals"

experiment = mlflow.set_experiment(EXPERIMENT)
mlflow.langchain.autolog()

import agent
import traffic as T
from eval_dataset import EVAL_DATASET

EXPERIMENT_ID = experiment.experiment_id
print(f"experiment    : {EXPERIMENT} (id {EXPERIMENT_ID})")
print(f"curated rows  : {len(EVAL_DATASET)}")


## Step 1 — What production traffic actually looks like

Real support traffic is heavily skewed: a few questions carry most of the volume, and
everything interesting lives in a long tail.

That shape is the whole reason sampling strategy matters. Look at the profile below before
sampling from it.

In [ ]:
# ============ THE SHAPE OF THE TRAFFIC ============
for note, share in T.traffic_profile().items():
    print(f"  {note:20} {share:6.1%}  {'#' * int(share * 60)}")

print()
print("'covered' = questions the curated dataset already tests.")
print("Everything else is either a paraphrase of those, or something no one wrote a test for.")


In [ ]:
# ============ RUN A BATCH OF TRAFFIC THROUGH THE AGENT ============
# This is the expensive cell in this notebook: one agent call per request.
REQUESTS = T.simulate_traffic(n=30, seed=7)
print(f"simulating {len(REQUESTS)} requests -- composition: {dict(Counter(n for _, _, n in REQUESTS))}\n")

for i, (query, customer_id, note) in enumerate(REQUESTS, 1):
    reply = agent.answer(query, customer_id=customer_id)
    print(f"{i:>3}. [{note:18}] {query[:56]}")
    print(f"     -> {reply[:100]}")


## Step 2 — Find the traces

`mlflow.search_traces` is the mining tool. Two things to remember about it:

- Filter fields need an `attributes.` or `tags.` prefix and **single-quoted** values.
- `return_type="list"` gives `Trace` objects; the default gives a DataFrame. **Dataset
  `merge_records` needs the list form**, so that argument matters more than it looks.

In [ ]:
# ============ SEARCH ============
import time

recent_cutoff = int((time.time() - 3600) * 1000)   # last hour, in milliseconds

traces = mlflow.search_traces(
    filter_string=f"attributes.status = 'OK' AND attributes.timestamp_ms > {recent_cutoff}",
    order_by=["attributes.timestamp_ms DESC"],
    max_results=200,
    return_type="list",          # list[Trace] -- required later by merge_records
)

print(f"traces found: {len(traces)}")


def trace_query(trace):
    """Pull the original question out of a trace's root span inputs."""
    root = trace.data._get_root_span()
    if root and root.inputs:
        return root.inputs.get("query", "")
    return ""


def trace_response(trace):
    root = trace.data._get_root_span()
    return (root.outputs if root else None) or ""


print("\nfirst few:")
for t in traces[:3]:
    print(f"  {t.info.trace_id[:16]}...  {trace_query(t)[:60]}")


## Step 3 — Sampling strategy decides what you learn

This is the part that separates a useful mined dataset from an expensive one that teaches
you nothing.

**Random sampling reproduces the distribution** — which means it mostly hands you the head:
the common questions your curated set already covers. You pay for LLM judges to re-confirm
what you already knew.

**Targeted sampling goes after the tail.** Several signals work; the cheapest useful one is
*novelty*: a query that doesn't resemble anything in your evaluation set is, by definition,
untested. No LLM call required — token overlap is enough to rank it.

In [ ]:
# ============ STRATEGY A: RANDOM SAMPLE ============
import random

rng = random.Random(11)
random_sample = rng.sample(traces, k=min(8, len(traces)))

curated_queries = {r["inputs"]["query"].lower() for r in EVAL_DATASET}

print("RANDOM SAMPLE OF 8")
already_covered = 0
for t in random_sample:
    q = trace_query(t)
    covered = q.lower() in curated_queries
    already_covered += covered
    print(f"  [{'already tested' if covered else 'NEW':>14}] {q[:62]}")

print(f"\n{already_covered}/{len(random_sample)} were questions the curated set already tests.")
print("That is the sampling strategy working exactly as designed -- and teaching nothing.")


In [ ]:
# ============ STRATEGY B: TARGETED BY NOVELTY ============
# Cheap, deterministic, explainable: how much does this query overlap, token-wise, with the
# closest thing already in the evaluation set? Low overlap = untested territory.

def tokens(text):
    return {w.strip(".,?!¿'\"").lower() for w in text.split() if len(w) > 2}


CURATED_TOKEN_SETS = [tokens(q) for q in curated_queries]


def novelty(query):
    """0.0 = identical to something we already test, 1.0 = nothing like it."""
    q = tokens(query)
    if not q:
        return 1.0
    best = max((len(q & c) / len(q | c) for c in CURATED_TOKEN_SETS), default=0.0)
    return 1.0 - best


scored = sorted(((novelty(trace_query(t)), t) for t in traces), key=lambda x: -x[0])

# Select by THRESHOLD, not by a fixed count. Taking "the top 8" pads the selection with
# whatever ranked highest among things you already test, once the genuinely novel material
# runs out -- and the tail is rare by construction, so it runs out fast. A threshold
# returns fewer rows on a quiet day and more after a traffic shift, which is the behaviour
# you want: the size of the selection should reflect how much new ground appeared.
NOVELTY_THRESHOLD = 0.40
MAX_SELECTED = 10

print(f"TARGETED SAMPLE (novelty >= {NOVELTY_THRESHOLD})")
seen, targeted = set(), []
for score, t in scored:
    q = trace_query(t)
    if not q or q.lower() in seen or score < NOVELTY_THRESHOLD:
        continue
    seen.add(q.lower())
    targeted.append((score, t))
    print(f"  novelty {score:.2f}  {q[:66]}")
    if len(targeted) >= MAX_SELECTED:
        break

print()
print(f"{len(targeted)} traces cleared the threshold out of {len(traces)} searched.")
print("A smaller, purer selection beats a padded one -- every row here costs human")
print("labelling time in Step 6, and a row you already test is pure overhead.")


Compare the two lists. The random sample is dominated by questions already under test;
the targeted one surfaces out-of-scope questions, a non-English request, a rambling
multi-part message, and a billing dispute.

**None of those appear in `eval_dataset.py`.** Several of them probe a behaviour the
curated set never tests at all — what the agent does when the knowledge base simply does
not contain the answer.

## Step 4 — Tag the traces you selected

Tagging is what makes selection reviewable and repeatable. The tag records *why* a trace
was picked, so a teammate (or you, in six weeks) can see the selection criteria rather than
inferring it from a list of trace IDs.

In [ ]:
# ============ TAG SELECTED TRACES ============
for score, t in targeted:
    mlflow.set_trace_tag(
        trace_id=t.info.trace_id,
        key="eval_candidate",
        value="novel_query",
    )

print(f"tagged {len(targeted)} traces with eval_candidate=novel_query")

# Tags are searchable, so selection becomes a reproducible query rather than a saved list.
tagged = mlflow.search_traces(
    filter_string="tags.eval_candidate = 'novel_query'",
    max_results=100,
    return_type="list",
)
print(f"re-fetched by tag: {len(tagged)} traces")


## Step 5 — Create a managed evaluation dataset

Two forms of `create_dataset`, and which one you use depends on where you're running:

```python
# OSS / local -- needs only a SQL-backed tracking store
create_dataset(name="...", experiment_id=[EXPERIMENT_ID])

# Databricks -- backs the dataset with a Unity Catalog table
create_dataset(uc_table_name="catalog.schema.telcoassist_mined")
```

The UC form additionally needs a Spark session (`DatabricksSession.builder.remote(serverless=True)`),
which is the usual cause of "no Spark session available" when people first try it.

A managed dataset differs from a Python list in ways that matter here: it is versioned, it
is attached to an experiment, it is queryable by tags, and `merge_records` **deduplicates by
input hash** — so re-mining the same traffic updates records instead of piling up copies.

In [ ]:
# ============ CREATE THE DATASET ============
from mlflow.genai.datasets import create_dataset, get_dataset

DATASET_NAME = "telcoassist_mined_from_traffic"

try:
    dataset = create_dataset(
        name=DATASET_NAME,
        experiment_id=[EXPERIMENT_ID],
        tags={"source": "simulated_production_traffic", "selection": "novelty_targeted"},
    )
    print(f"created dataset '{DATASET_NAME}'")
except Exception as exc:
    # Re-running this notebook shouldn't fail just because the dataset already exists.
    print(f"create failed ({type(exc).__name__}), loading existing dataset instead")
    dataset = get_dataset(name=DATASET_NAME)

print(f"dataset_id: {dataset.dataset_id}")


In [ ]:
# ============ MERGE THE TAGGED TRACES IN ============
dataset = dataset.merge_records(tagged)

df = dataset.to_df()
print(f"records in dataset: {len(df)}")
print(f"columns: {list(df.columns)}")


## Step 6 — The manual part: expectations, with provenance

Here is where the circularity warning from the top becomes concrete. The mined records have
**inputs** and they have the agent's **outputs**, but they have no ground truth — and the
outputs cannot become the ground truth without making the dataset self-fulfilling.

So a human decides what each of these should have produced. `mlflow.log_expectation`
attaches that judgement to the trace, and `AssessmentSource` records **who decided**.

That provenance field is not bureaucracy. Six months from now, "is this expectation a
domain expert's considered judgement, or something a script inferred?" determines how much
weight a failure against it deserves — and whether it is safe to use for aligning a judge
(Phase 7) or optimising a prompt (Phase 8).

In [ ]:
# ============ ATTACH GROUND TRUTH, WITH A RECORDED SOURCE ============
from mlflow.entities import AssessmentSource, AssessmentSourceType

HUMAN = AssessmentSource(
    source_type=AssessmentSourceType.HUMAN,
    source_id="sourav@example.com",     # who made this judgement
)

# The knowledge base genuinely contains nothing about family plans, service pauses, 5G
# tiers, or student discounts. The correct behaviour is to say so and offer a human --
# NOT to improvise a plausible-sounding answer.
OUT_OF_SCOPE_MARKERS = ("family plan", "pause my service", "5g", "student discount")

labelled = 0
for t in tagged:
    q = trace_query(t).lower()
    if any(marker in q for marker in OUT_OF_SCOPE_MARKERS):
        mlflow.log_expectation(
            trace_id=t.info.trace_id,
            name="guidelines",
            value=[
                "The response must state that this information is not available.",
                "The response must offer to connect the customer with a human agent.",
                "The response must not invent plan names, prices, or eligibility rules.",
            ],
            source=HUMAN,
        )
        labelled += 1
        print(f"  labelled (abstention expected): {trace_query(t)[:60]}")

print(f"\n{labelled} traces given human-authored expectations")
print(f"{len(tagged) - labelled} still unlabelled -- mining found them, a human must still judge them")


That last line is the honest accounting for this phase. Mining surfaced eight interesting
cases in seconds; turning them into *tests* took a person deciding what each one should do,
and most of them are still waiting for that.

This is why "just use production data" is a half-truth. The half it gets right — knowing
which inputs matter — is genuinely the expensive half to guess.

In [ ]:
# ============ MERGE SEMANTICS: SAME INPUTS UPDATE, THEY DON'T DUPLICATE ============
probe = {
    "inputs": {"query": "Do you offer family plans for four lines?", "customer_id": None},
    "expectations": {"expects_tool_call": False},
}

before = len(dataset.to_df())
dataset = dataset.merge_records([probe])
after_first = len(dataset.to_df())

# Same inputs again, with an additional expectation -- expectations merge, no new row.
dataset = dataset.merge_records([{
    "inputs": probe["inputs"],
    "expectations": {"must_offer_human": True},
}])
after_second = len(dataset.to_df())

print(f"records before      : {before}")
print(f"after first merge   : {after_first}")
print(f"after repeat merge  : {after_second}  <- unchanged: matched on input hash")
print()
print("Re-mining overlapping traffic is therefore safe: records update rather than pile up,")
print("and expectations accumulate instead of overwriting one another.")


## Step 7 — What the mined data actually revealed

The curated dataset has twelve rows and not one of them asks a question the knowledge base
cannot answer. **Abstention was never tested** — so nobody knew whether the agent says "I
don't have that" or confidently invents a family plan.

That gap wasn't found by thinking harder about the agent. It was found by looking at
traffic. And closing it needs a scorer that didn't exist before.

In [ ]:
# ============ A SCORER THE MINED DATA MOTIVATED ============
from mlflow.genai.scorers import ExpectationsGuidelines, Guidelines, Safety

abstains = Guidelines(
    name="abstains_when_unsupported",
    guidelines=(
        "If the support articles provided to the agent do not contain the information "
        "needed to answer the question, the response must say the information is not "
        "available and offer to connect the customer with a human agent. It must not state "
        "specific plan names, prices, or eligibility rules that are absent from the "
        "articles. If the articles do cover the question, this guideline is automatically "
        "satisfied."
    ),
)

OUT_OF_SCOPE_EVAL = [
    {"inputs": {"query": q}}
    for q, _, _, note in T.TRAFFIC_MIX
    if note == "out-of-scope"
]

print(f"evaluating abstention on {len(OUT_OF_SCOPE_EVAL)} out-of-scope questions:")
for r in OUT_OF_SCOPE_EVAL:
    print(f"  - {r['inputs']['query']}")


In [ ]:
# ============ RUN IT ============
with mlflow.start_run(run_name="mined_out_of_scope_abstention"):
    mlflow.set_tag("dataset_source", "mined_from_traffic")

    abstention_results = mlflow.genai.evaluate(
        data=OUT_OF_SCOPE_EVAL,
        predict_fn=agent.answer,
        scorers=[abstains, Safety()],
    )

for key, value in sorted(abstention_results.metrics.items()):
    printable = f"{value:.3f}" if isinstance(value, (int, float)) else str(value)
    print(f"  {key:44} {printable:>8}")

print()
print("Whatever this score is, note that Phases 2, 3 and 5 could not have produced it --")
print("there was no row in the curated dataset that exercised this path.")


In [ ]:
# ============ INSPECT WHAT THE AGENT ACTUALLY SAID ============
traces_df = mlflow.search_traces(run_id=abstention_results.run_id)

for _, row in traces_df.iterrows():
    print(f"Q: {str(row['request'])[:80]}")
    print(f"A: {str(row['response'])[:240]}")
    for a in row["assessments"] or []:
        name = a.get("assessment_name") or a.get("name") if isinstance(a, dict) else getattr(a, "name", None)
        if name == "abstains_when_unsupported":
            fb = a.get("feedback", {}) if isinstance(a, dict) else getattr(a, "feedback", None)
            val = fb.get("value") if isinstance(fb, dict) else getattr(fb, "value", None)
            rationale = a.get("rationale") if isinstance(a, dict) else getattr(a, "rationale", None)
            print(f"   -> {val}: {str(rationale)[:180]}")
    print()


## Step 8 — Close the loop

A discovery that stays in a mined dataset helps once. Promote it into the curated set and
it protects every future release:

1. **Add rows** to `eval_dataset.py` for the abstention cases, with human-authored
   expectations — so Phase 5's regression gate covers them from now on.
2. **Add `abstains_when_unsupported`** to the standard scorer set.
3. **Add a gate** for it in `QUALITY_GATES` — remembering Phase 5's lesson that a scorer
   with no gate entry blocks nothing.
4. **Keep mining.** Traffic keeps changing; this is a loop, not a task.

The curated and mined datasets play different roles and both are needed: the curated set is
*fixed*, which is what makes version comparison valid, while the mined set is *current*,
which is what keeps the curated set from going stale.

## Key takeaways

- **Hand-written datasets are structurally blind.** They contain what their author imagined,
  and the gap between that and real traffic is invisible from the inside.
- **Mining gives you inputs, not ground truth.** Using the agent's own outputs as expected
  answers produces a dataset it passes by construction and makes regression undetectable —
  you'd have frozen today's bugs as the definition of correct.
- **Sampling strategy is the whole game.** Random sampling reproduces the head of a skewed
  distribution and re-confirms what you already test. Novelty-based targeting — cheap,
  deterministic, no LLM needed — goes after the part you don't.
- **Tag selections rather than saving lists of IDs**, so the criteria stay visible and the
  selection stays reproducible.
- **Record who authored each expectation.** `AssessmentSource` distinguishes a domain
  expert's judgement from a script's inference, which matters enormously in Phases 7 and 8
  where that ground truth trains a judge and drives prompt optimisation.
- **`merge_records` deduplicates by input hash and merges expectations**, so re-mining
  overlapping traffic is safe and labelling can accumulate over time.
- **Managed datasets need a SQL-backed store** — the same constraint as the Prompt Registry,
  and another reason `file://` tracking doesn't get you far.
- **Mined and curated sets are complements**: fixed for comparison, current for discovery.

**Next: Phase 6 — online evaluation. Everything so far has been offline, judging a fixed
dataset. Production monitoring scores live traffic continuously, on a sample, and is how
you find out what even your mined dataset didn't contain yet.**